# Logical and Physical Plans: Deep Reading and Plan Forensics

**Module**: 07_spark_sql_internals  
**Notebook**: 02_logical_and_physical_plans.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Step-by-step guide to reading explain() outputs: Project, Filter, Join, Aggregate, Exchange, and Sort nodes.
- **Why does this topic exist?**: Reading execution plans is the primary method to detect bottlenecks before running long jobs.
- **What problem does it solve?**: Blindly running queries without knowing if they will shuffle or broadcast.
- **Where is it used in real-world Data Engineering?**: Query tuning in production environments.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Catalyst basics, DataFrame API |
| **Required depth** | Deep working knowledge: can read plans and identify the operator that dominates |
| **Data Engineer** | Read logical and physical plans and map operators to runtime metrics. |
| **Senior Data Engineer** | Use plan forensics to explain performance regressions. |
| **Spark specialist** | Reason about plan shape, exchanges, and operator-specific costs. |
| **Next topic** | `07_spark_sql_internals/03_tungsten_and_whole_stage_codegen.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A plan is a tree of operators. The logical plan describes what to compute; the physical plan describes how.
Reading both lets you see whether a filter was pushed, which join strategy was chosen, where exchanges occur, and
which operator is likely to dominate. Plan reading is the fastest way to move from "it is slow" to a hypothesis.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Logical plan** | What to compute | Resolved tree of relational operators |
| **Physical plan** | How to compute it | Executable operators such as Scan, Join, Exchange |
| **Exchange** | Shuffle boundary | Redistributes data by partitioning expression |
| **Scan** | Source read | Shows format, filters, and partition pruning |
| **Join operator** | Join implementation | BroadcastHashJoin, SortMergeJoin, etc. |
| **AQE plan** | Runtime-adjusted plan | May change after materializing shuffle statistics |

### How it works, step by step

1. Start from the final output and read upward.
2. Identify scans, filters, projections, joins, aggregations, and exchanges.
3. Check which filters were pushed into the scan.
4. Check the join strategy and the size estimates.
5. Count exchanges and stage boundaries.
6. Compare the plan with the Spark UI stages and task metrics.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Reading Plan Operators

- `FileScan`: Table reader (look for PushedFilters and PartitionFilters).
- `Project`: Column selection / expression calculation.
- `Filter`: Row filtering.
- `Exchange`: Shuffle boundary.
- `HashAggregate` / `SortAggregate`: Grouping operation.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Plans_Forensics_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
p = orders.filter("status = 'Completed'").groupBy("channel").count().orderBy("count")
p.explain("formatted")

---

## Experiment: Extended Plan Inspection

- **1. What are we testing?**: Print explain('extended') to compare analyzed vs optimized plans.
- **2. What do we expect?**: Optimized plan is noticeably simpler.

In [ ]:
p.explain('extended')

**3. What happened?**: Unneeded projections and expressions were collapsed.  
**4. Why did it happen?**: Catalyst removed redundant nodes.  
**5. Practical Engineering Lesson**: Optimized plans show the true operations that will run.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Regression after a library upgrade

- **Industry**: SaaS analytics
- **Scenario**: A query becomes three times slower after an upgrade without code changes.
- **Problem**: The physical plan changed from a broadcast join to a sort-merge join, adding a large shuffle.
- **Why this concept is relevant**: Comparing plans before and after isolates the strategy change from data changes.
- **Implementation idea**: Capture plans in CI for critical queries and diff the operators and exchanges.
- **Expected result**: A targeted fix such as refreshing statistics or adjusting the join.
- **Alternative**: Pin the Spark version.
- **Trade-offs**: Pinning delays the upgrade; plan diffing addresses the root cause.

### Use Case 2: Unexpected full scan

- **Industry**: Media
- **Scenario**: A dashboard query scans the entire event table despite a date filter.
- **Problem**: The filter is applied after a union or a view that obscures the partition column.
- **Why this concept is relevant**: Reading the plan shows the scan lacks partition filters, so the predicate is not reaching the source.
- **Implementation idea**: Rewrite the query to filter each branch before the union, then verify partition filters in the scan.
- **Expected result**: A scan limited to the requested dates.
- **Alternative**: Materialize a date-partitioned summary table.
- **Trade-offs**: A summary table speeds reads but adds pipeline maintenance.

---

## Business Value

- Faster root-cause analysis for slow queries.
- Fewer speculative tuning changes.
- Better code and SQL review practices.
- Stable performance across upgrades.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Read plans from the bottom up and identify exchanges first.
- Check `PushedFilters` and `PartitionFilters` on scans.
- Compare estimated and actual row counts when available.
- Capture plans for critical queries before upgrades.
- Use `explain('formatted')` for readability and `explain('cost')` when statistics exist.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Diff two plans

**Steps**

1. Run a join query and save `explain('formatted')`.
2. Change the filter selectivity or disable broadcast, then save the new plan.
3. Compare join strategy, exchanges, and scan filters.

**What to observe**: Small query changes can alter join strategy and shuffle volume.

**Healthy result**: You can explain each difference with a specific rule or statistic.

**Problematic result**: The plan changes but runtime does not, indicating the bottleneck is elsewhere.

**Engineering takeaway**: Plans explain strategy; metrics explain cost.

---

## Trade-offs and Alternatives

### Advantages

- Precise diagnosis
- No extra tooling
- Works for SQL and DataFrames

### Limitations

- Verbose output
- Estimates can be misleading
- AQE plans change at runtime

### When to use it

- Performance reviews
- Upgrade testing
- Join debugging

### When not to use it

- Reading only the final plan
- Ignoring runtime metrics
- Treating estimates as facts

### Alternatives inside Spark

- Logical plan
- Physical plan
- AQE
- Spark UI

### Alternatives outside Spark

- Query profilers
- Warehouse explain plans
- Plan visualizers

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Find Root Operator
**Problem Statement**: Identify the top-level operator in a query plan.  
**Input / Context**: p

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Count Projects
**Problem Statement**: Count Project operators in plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Cost Plan
**Problem Statement**: Run explain('cost') and check statistics.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The plan that looks fine but runs slowly

**Scenario**: The physical plan shows a broadcast join and pushed filters, yet the job is slow.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check task duration and input size per task.
2. Look for skew in the probe side.
3. Inspect source read throughput and file count.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Find Root Operator):
```python
print('Top operator is SortExec or AdaptiveSparkPlan')
```

#### Solution 2 (Count Projects):
```python
print('Inspected plan strings')
```

#### Solution 3 (Cost Plan):
```python
p.explain('cost')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Plan Reading**: Scan -> Filter -> Project -> Exchange -> Aggregate -> Sort.

### Essential Technical Terms
- `ExecutedPlan`: Physical plan after code generation.
- `AST`: Tree of relational operators.

### Common Engineering Mistakes
- **Confusing Parsed with Optimized**: Parsed plan still contains unverified syntax.

### Interview Questions on This Topic
- **Q: How do you view all 4 plan stages?**
  - *A*: Call df.explain(True) or df.explain('extended').


### More Interview Questions

- **Q: What is the difference between a logical and a physical plan?**
  - *A*: A logical plan describes the relational operations to compute; a physical plan specifies the executable operators and data movement, such as scans, joins, and exchanges.
- **Q: Why count exchanges in a plan?**
  - *A*: Each exchange is a shuffle and a stage boundary, usually the most expensive part of the query; fewer or smaller exchanges often mean less I/O and network traffic.

### What to Study Next
Open 03_tungsten_and_whole_stage_codegen.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')